### data ingetion


In [17]:
from langchain_core.documents import Document

In [18]:
doc=Document(
    page_content="this is where content resides q",
    metadata={
        "source":"example.txt",
        "pages":"1",
        "author":"Ishan",
        "date_created":"2026-8-7"
    }
)

In [19]:

sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [20]:
##text loader
from langchain_community.document_loaders  import TextLoader

In [21]:
loader=TextLoader("../data/text_files/python_intro.txt")
dox=loader.load()
print(dox)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [22]:
#dirctory loadeer
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":'utf-8'},
    show_progress=True
)

doc=dir_loader.load()
print(doc)

100%|██████████| 2/2 [00:00<00:00, 1192.92it/s]

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'), Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised

In [23]:
import numpy as np
import chromadb
import uuid
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [24]:
class EmbeddingManager:
    """handles document embedding generation using sentenceTransformer"""
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """load the sentenseTransformer model"""
        try:
            print("loading the model:",self.model_name)
            self.model=SentenceTransformer(self.model_name)
            print("model loaded with embedding : ",self.model.get_embedding_dimension())
        except Exception as e:
            print("error loading the model",e)
            raise
    def generate_embedding(self,text:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("model not init")
        print(f"generateing embedding for {len(text)} texts")
        embedings=self.model.encode(text,show_progress_bar=True)
        print(f"embeding generated with shape {embedings.shape}")
        return embedings
##initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


loading the model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4864.14it/s]


model loaded with embedding :  384


### vector store


In [25]:
import os

In [26]:

class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persisit_dir:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_dir=persisit_dir
        self.client=None
        self.collection=None
        self.embedding_matrix = None
        self.document_texts: List[str] = []
        self.doc_ids: List[str] = []
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_dir)
            self.collection=self.client.get_or_create_collection(self.collection_name,metadata={"description":"PDF embeddings for RAG"})
            print(f"collection initialized with name {self.collection_name}")
            print(f"document count in collection {self.collection.count()}")
        except Exception as e:
            print("error creating db collection {e}")
            raise
    def add_document(self,documents:List[Any],embedding:np.ndarray):
        if len(documents)!=len(embedding):
            raise ValueError("number of documet must be same")
        print(f"adding {len(documents)} to vector database")
        ids=[]
        metadatas=[]
        document_text=[]
        embedding_list=[]
        for i,(doc,embeds)in enumerate(zip(documents,embedding)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['document_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            document_text.append(doc.page_content)
            embedding_list.append(embeds.tolist())
        try:
            self.collection.add(ids,embedding_list,metadatas,documents=document_text)
            self.embedding_matrix = np.asarray(embedding_list)
            self.document_texts = document_text
            self.doc_ids = ids
            print(f"succefully added {len(documents)} documents to vector store")
            print(f"total documents in collection {self.collection.count()}")
        except Exception as e:
            print("error adding into vector store : ",e)
            raise
    def get_all_embeddings(self):
        if self.embedding_matrix is None:
            raise ValueError("No embeddings loaded in memory")
        return self.embedding_matrix

    def get_all_documents(self):
        return self.doc_ids, self.document_texts, self.embedding_matrix
vectorstore=VectorStore()
vectorstore

collection initialized with name pdf_documents
document count in collection 17


In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=100,chunk_overlap=20):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [28]:
chunks=split_documents(doc)
chunks

Split 2 documents into 15 chunks

Example chunk:
Content: Python Programming Introduction...
Metadata: {'source': '../data/text_files/python_intro.txt'}


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Created by Guido van Rossum and first released in 1991, Python has become one of the most popular'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='programming languages in the world.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Key Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='- Strong community support'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='P

In [29]:
chunks

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Created by Guido van Rossum and first released in 1991, Python has become one of the most popular'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='programming languages in the world.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Key Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='- Strong community support'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='P

In [30]:
texts=[docu.page_content for docu in chunks]
texts

['Python Programming Introduction',
 'Python is a high-level, interpreted programming language known for its simplicity and readability.',
 'Created by Guido van Rossum and first released in 1991, Python has become one of the most popular',
 'programming languages in the world.',
 'Key Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility',
 '- Strong community support',
 'Python is widely used in web development, data science, artificial intelligence, and automation.',
 'Machine Learning Basics',
 'Machine learning is a subset of artificial intelligence that enables systems to learn and improve',
 'from experience without being explicitly programmed. It focuses on developing computer programs',
 'that can access data and use it to learn for themselves.',
 'Types of Machine Learning:\n1. Supervised Learning: Learning with labeled data',
 '2. Unsupervised Learning: Finding patterns in unlabeled data',
 '3. Reinforcement Learning: Learning throu

In [31]:
##generate embding
embedding=embedding_manager.generate_embedding(texts)
##store into vector db
vectorstore.add_document(chunks,embedding=embedding)

generateing embedding for 15 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.80it/s]

embeding generated with shape (15, 384)
adding 15 to vector database
succefully added 15 documents to vector store
total documents in collection 32


In [32]:
class RAGRetrieval:
    def __init__(self, vectorstore: VectorStore, embeddingmanager: EmbeddingManager):
        self.vector_store = vectorstore
        self.embeddingmanager = embeddingmanager

    def retrieve(self, query: str, k: int = 3, score_threshold: float = 0.0):
        """Retrieve the top-k matching chunks for a query using cosine similarity."""
        qembeds = self.embeddingmanager.generate_embedding([query])
        _, documents, embeddings = self.vector_store.get_all_documents()
        similarity_scores = cosine_similarity(qembeds, embeddings)[0]
        top_indices = np.argsort(similarity_scores)[::-1][:k]

        return [
            {
                "id": self.vector_store.doc_ids[i],
                "document": documents[i],
                "score": float(similarity_scores[i])
            }
            for i in top_indices
            if similarity_scores[i] >= score_threshold
        ]
rag=RAGRetrieval(vectorstore,embedding_manager)
relateddoc=rag.retrieve("what is the attention in rag")
relateddoc

generateing embedding for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 139.07it/s]

embeding generated with shape (1, 384)


[{'id': 'doc_76e32497_9',
  'document': 'from experience without being explicitly programmed. It focuses on developing computer programs',
  'score': 0.0771914624383297},
 {'id': 'doc_b8049602_14',
  'document': 'Applications include image recognition, speech processing, and recommendation systems',
  'score': 0.07067231194244161},
 {'id': 'doc_f1288885_5',
  'document': '- Strong community support',
  'score': 0.05453676250889762}]

In [ ]:
import os
os.environ.get('OPENAI_API_KEY')